### This notebook is used for small-scale PRAS simulation to run and save data. For full-scale simulation, please refer to the pras_simulation Julia script.
* It read the Matpower data file that has been constructed from the Security Constraint multi-network run from 2024 to 2044. The last network (2044) is constructed and generates the file (nw_2044_prod_v1.m). This is used as the base data file, and on top of that, storage and generators are added.
* Time series data with thermal generators' expiry dates are implemented.
* Future storage and generation with time series data added.
* Storage time series is not implemented in the PowerSystems; the time series data with binary 0 and 1, indicating the power value with the starting time (1) and multiplying with the actual power, is directly added to the pras storage matrix data.
* Every expiry time is hard-coded with the 1st of Dec with the respective year added for expiry thermal generators, and similarly for storage time series data.
* It has an additional 23 buses and has an additional 18 new "Connection "  lines

In [4]:
using PowerSystems
using CSV
#using HDF5
using DataFrames
using Dates
using PRAS
using JuMP
#using Plots
using PowerModels
using PowerSimulations
using PowerSystemCaseBuilder
#using PowerNetworkMatrices
using StorageSystemsSimulations
#using HiGHS
using TimeSeries
using Dates: DateTime
using Logging
using DataStructures
using HydroPowerSimulations
using HiGHS

const PSI = PowerSimulations

import InfrastructureSystems
const IS = InfrastructureSystems
#import InfrastructureModels as _IM

Logging.disable_logging(Logging.Warn)
PowerModels.silence()

import DataStructures: SortedDict
const PSY = PowerSystems
#const PSI = PowerSimulations
#const PSB = PowerSystemCaseBuilder
#const PNM = PowerNetworkMatrices

ENV["TMPDIR"] = "/tmp"


[info | PowerModels]: Suppressing information and warning messages for the rest of this session.  Use the Memento package for more fine-grained control of logging.


"/tmp"

In [5]:
include("src/utility.jl")
include("src/pras_utility.jl")

save_pras_lines_limits_info (generic function with 1 method)

In [241]:
#pm_data = PSY.PowerModelsData(file_path)

In [148]:
file_path = joinpath(pwd(), "../../../data", "sc_data", "snem_step_change_base_case_2044_2.m")
sys, pm_data, base_storage_data = initialise_system_v3(file_path)
println("system loaded")

system loaded


### Load the static network 2044 dataset and initialise the system

### Saving some system data for analysis and preparing for report
ofile = "data/output/nem_act_power_load_gen_v_44.csv"
file_path = joinpath(pwd(), "data", "sc_data", "nw_2044_prod_v1.m")
sys_44, pm_data_44, base_storage_data_44 = initialise_system_v3(file_path)
lg_dict_44, region_base_data_44 = get_load_gen_storage_system(sys_44, ofile)
ofile = "data/output/nem_act_power_load_gen_v_24.csv"
file_path = joinpath(pwd(), "data", "sc_data", "nw_2024_prod_v1.m")
sys_24, pm_data_24, base_storage_data_24 = initialise_system_v3(file_path)
lg_dict_24, region_base_data_24 = get_load_gen_storage_system(sys_24, ofile)
println("done")

In [147]:
#ofile = "data/output/nem_base_info.csv"
#lg_dict, region_base_data = get_load_gen_storage_system(sys, ofile)

### Select scenario type. For ex. ST, MT, LT and SUM_ED
## New scenario from SC - Multinetworks LT_BASE, LT_BEST, LT_TYP, LT_WORST 

In [ ]:
scenario = "LT_BASE"

location = joinpath(pwd(), "data", "sc_data")
add_baseload(sys, scenario, location)

if contains(scenario, "LT")
    add_future_gen(sys, "LT", location)
    add_future_storage(sys, "LT", location)
    file_path = joinpath(location, "future_gen_thermal_exp_pp.csv")
    add_future_status_therm_stor(sys, "LT", file_path, "gen")
    file_path = joinpath(location, "future_storage_pp.csv")
    add_future_status_therm_stor(sys, "LT", file_path, "storage")
else    
    add_future_gen(sys, scenario, location)
    add_future_storage(sys, scenario, location)
    file_path = joinpath(location, "future_gen_thermal_exp_pp.csv")
    add_future_status_therm_stor(sys, scenario, file_path, "gen")
    file_path = joinpath(location, "future_storage_pp.csv")
    add_future_status_therm_stor(sys, scenario, file_path, "storage")
end

println("Scenario choosen: ", scenario)
if scenario == "ST"
    time_series_data = built_load_one_yearly_TSdata(sys)
elseif scenario == "MT"
    time_series_data = built_load_six_yearly_TSdata(sys)    
elseif contains(scenario, "LT")
    time_series_data = built_load_twenty_years_TSdata(sys)
elseif scenario == "SUM_ED"
    time_series_data = built_load_SUM_ED_TSdata(sys)
end
println("Added thermal generators and storage time series data")

In [62]:
sys

Property,Value
Name,snem_15_regions
Description,"15-regions SNEM ACDC model, this representation is based on clustering data from 15 regions"
System Units Base,SYSTEM_BASE
Base Power,100.0
Base Frequency,50.0
Num Components,9873
Type,Count
ACBus,2042
Arc,2480
Area,15


In [24]:
#If needed to save the scenario specific system then save it by calling this function
#save_newly_built_system(sys, scenario)

### Convert into PRAS model

In [63]:
using SiennaPRASInterface

const DEFAULT_DEVICE_MODELS_ST = [
        DeviceRAModel(PSY.Line, LinePRAS),
        DeviceRAModel(PSY.TwoTerminalHVDCLine, LinePRAS),
        DeviceRAModel(PSY.StaticLoad, StaticLoadPRAS),
        DeviceRAModel(PSY.ThermalGen, GeneratorPRAS),
        DeviceRAModel(PSY.RenewableGen, GeneratorPRAS),
        DeviceRAModel(PSY.HydroDispatch, GeneratorPRAS),
        DeviceRAModel(PSY.EnergyReservoirStorage, EnergyReservoirLossless)
    ]   
const DEFAULT_TEMPLATE = RATemplate(PSY.Area, DEFAULT_DEVICE_MODELS_ST)

gps = generate_pras_system(sys, DEFAULT_TEMPLATE)
println("PRAS model - GPS - is created for scenario:", scenario)

PRAS model - GPS - is created for scenario:LT_BASE


In [27]:
# This is required to add time series storage data here, as importing the scaling values in the
# Power Systems not being implemented
for (i, kk) in enumerate(gps.storages.names)
    comp_data = get_component(EnergyReservoirStorage, sys, kk)
    ts_data = get_time_series(SingleTimeSeries, comp_data, "max_active_power")
    ts_data = ts_data.data
    gps.storages.energy_capacity[i,:] = gps.storages.energy_capacity[i,:] .* values(ts_data)
    gps.storages.charge_capacity[i,:] = gps.storages.charge_capacity[i,:] .* values(ts_data)
    gps.storages.discharge_capacity[i,:] = gps.storages.discharge_capacity[i,:] .* values(ts_data)
end
println("done")

done


### Save the pras lines and line capacity information

In [ ]:
fname = "pras_lines_" * scenario * ".csv"
ofile = joinpath(pwd(), "data", "output", fname)
pras_lines_df = get_pras_lines(gps)
CSV.write(ofile, pras_lines_df)
location = joinpath(pwd(), "data", "output")
ints_core = save_line_capacity(gps, location, scenario)

### Running Monte Carlo simulation

In [ ]:
# Do the SMC with a small sample or use the script for larger sample
resultspecs = (Shortfall(), Surplus(), Flow(), Utilization(), StorageEnergy(),
    GeneratorStorageEnergy(),GeneratorAvailability(), LineAvailability(), StorageAvailability(),
    GeneratorStorageAvailability())
samples = 100
println("scenario: ", scenario,",", "sample size: ", samples)
println("SMC starting at ", now())
smallsample = SequentialMonteCarlo(samples=samples, seed=123, threaded=false)
shortfall_rs, surplus_rs, flow_rs, util_rs, energy, gs_energy, ga, la, sa, gsa =
        assess(gps, smallsample, resultspecs...)
println("PRAS simulation done for sample size:", samples,",", now())
odir = joinpath(pwd(), "data", "pras_metrics_output", "shortfall_data", scenario)
#odir = joinpath(pwd(), "../../../data", "ISP_data", "pras_metrics_output", "shortfall_data", scenario)
mkpath(odir)
saveshortfall(shortfall_rs, gps, odir)
println("PRAS result is saved")

scenario: LT_BASE,sample size: 100
SMC starting at 2026-04-22T10:01:52.770


In [115]:
odir = joinpath(pwd(), "data", "pras_metrics_output", "shortfall_data", scenario)
use_df = save_shortfall_eue_metrics(gps, shortfall_rs, scenario, odir)
println("Unserved energy during shortfall time extracted")
save_shortfall_time_series_data(scenario, shortfall_rs, gps, flow_rs, util_rs, odir)
save_generator_storage_data(ga, gps, energy, scenario, odir)

Unserved energy during shortfall time extracted
Saved shortfall data
pras time series metrics calculated
Saved PRAS time series metrics data
pras regional loads calculated
pras metrics data saved
Saved energy data


### Doing Time-sequential simulation with Unit Commitment for ST scenario

In [198]:
# Use ST scenario only; other scenarios will not work.

steps = 7
pasa = "ST"
name = "SNEM_" * pasa
sim_st = run_simulation(sys, pasa, name, steps)

In [ ]:
#save data and process using Python notebook
# "power_simulation_res.ipynb" from the analysis folder 
merged_df = extract_data(sim_st)
ofile = joinpath(pwd(), "data", "output", "seq_sim_output_test_case.csv")
CSV.write(ofile, merged_df)
